# Kaggle CPU-only benchmark notebook for Maya-Voice-Os

This notebook is designed for a CPU-only Kaggle environment (roughly 13-16 GB RAM) and is meant to benchmark the Maya-Voice-Os pipeline end-to-end:

- ASR -> LLM routing -> TTS
- pytest checks for CLI eval logic
- latency benchmark via `maya-eval --suite latency --calls 30 --concurrency 2`

Important notes:
- This is intentionally CPU-only; it does not use a GPU.
- If you have a wheel URL instead of a GitHub repo URL, replace the install URL in the next code cell.
- If you want to change the number of calls or concurrency, edit the `CALLS` and `CONCURRENCY` variables below.
- The full output is saved to `benchmark_output.txt` so you can download it from Kaggle.

In [ ]:
import os
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("This notebook is designed for CPU-only Kaggle sessions (~13-16 GB RAM).")
print("No GPU is required.")

# --- Adjust these if needed ---
MAYA_GIT_URL = "https://github.com/Jugalt-iam/Maya-Voice-Os.git"
MAYA_WHEEL_URL = ""   # Example: "https://your-bucket.example.com/maya_voice_os.whl"
CALLS = 30
CONCURRENCY = 2
THRESHOLD_MS = 2500.0
REPO_DIR = "/kaggle/working/Maya-Voice-Os"
OUTPUT_FILE = "/kaggle/working/benchmark_output.txt"

# Install the minimum tooling we need for this benchmark.
# `pytest` is required for tests/test_cli_eval.py.
# `git` is needed if you install from GitHub instead of a wheel.
for cmd in [
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", "pytest"],
]:
    print(f"\n>>> Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise SystemExit(f"Dependency install failed: {' '.join(cmd)}")

print("\nDependency install completed.")
print(f"GitHub repo URL: {MAYA_GIT_URL}")
print(f"Wheel URL: {MAYA_WHEEL_URL or '(not used; using GitHub repo)'}")
print(f"Latency settings: calls={CALLS}, concurrency={CONCURRENCY}, threshold_ms={THRESHOLD_MS}")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Maya-Voice-Os")
OUTPUT_FILE = Path("/kaggle/working/benchmark_output.txt")

# If you have a wheel URL, set MAYA_WHEEL_URL in the previous cell.
# Otherwise this will clone the GitHub repo and install it in editable mode.
# This keeps the benchmark reproducible and easy to debug from Kaggle.
if MAYA_WHEEL_URL:
    print(f"Installing from wheel: {MAYA_WHEEL_URL}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", MAYA_WHEEL_URL],
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise SystemExit(f"Wheel install failed: {MAYA_WHEEL_URL}")
else:
    if REPO_DIR.exists():
        print(f"Removing existing repo directory: {REPO_DIR}")
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)

    print(f"Cloning repo: {MAYA_GIT_URL}")
    subprocess.run(["git", "clone", "--depth", "1", MAYA_GIT_URL, str(REPO_DIR)], check=True)

    os.chdir(REPO_DIR)
    print(f"Installing Maya-Voice-Os from: {REPO_DIR}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "-e", "."],
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise SystemExit(f"Package install failed for {REPO_DIR}")

print("Maya-Voice-Os install complete.")
print("Checking for CLI entry points...")
for tool in ["pytest", "maya-eval", "maya-voice-os"]:
    print(f"{tool}: {'FOUND' if shutil.which(tool) else 'MISSING'}")

In [ ]:
import os
import shutil
import subprocess

REPO_DIR = "/kaggle/working/Maya-Voice-Os"
OUTPUT_FILE = "/kaggle/working/benchmark_output.txt"

# Make sure the repo is in the working directory for test discovery.
if not os.path.exists(REPO_DIR):
    raise SystemExit(f"Repository not found at {REPO_DIR}. Install the package first.")

os.chdir(REPO_DIR)
with open(OUTPUT_FILE, "w", encoding="utf-8") as fh:
    fh.write("")

def run_logged(command, label):
    print(f"\n\n===== {label} =====")
    print(f"$ {command}")
    result = subprocess.run(command, shell=True, text=True, capture_output=True)
    combined = (result.stdout or "") + (result.stderr or "")
    print(combined)
    with open(OUTPUT_FILE, "a", encoding="utf-8") as fh:
        fh.write(f"\n\n===== {label} =====\n")
        fh.write(f"$ {command}\n")
        fh.write(combined)
    if result.returncode != 0:
        raise SystemExit(f"{label} failed with exit code {result.returncode}")

# Fail fast if the required tools are missing.
if shutil.which("pytest") is None:
    raise SystemExit("ERROR: pytest not found in PATH. Install it first with `python -m pip install pytest`.")
if shutil.which("maya-eval") is None:
    raise SystemExit("ERROR: `maya-eval` not found in PATH. Ensure Maya-Voice-Os was installed successfully.")

# Run the CLI eval test suite
run_logged("pytest -q tests/test_cli_eval.py", "pytest -q tests/test_cli_eval.py")

# Run the benchmark suite.
# Adjust these values in the setup cell if you want a different trial size or concurrency.
run_logged("maya-eval --suite latency --calls 30 --concurrency 2", "maya-eval --suite latency --calls 30 --concurrency 2")

print(f"\nAll benchmark output saved to: {OUTPUT_FILE}")
print("You can download this file from the Kaggle notebook output panel.")

# How to interpret the results + next steps

This notebook prints the benchmark output directly and also writes the full logs to `benchmark_output.txt`.

What to look for:
- P50 / P95 / P99 first-response latency
- ASR / LLM / TTS stage breakdown
- max concurrency before crossing the latency threshold
- fixed-test pass rate

Key ideas:
- This benchmark is intentionally conservative and CPU-only.
- It avoids heavy parallelism, which keeps it compatible with typical Kaggle CPU instances.
- If the benchmark is too slow or the output is unstable, lower `CALLS` or `CONCURRENCY` in the setup cell.
- If the repo changes or you want to test a different branch, replace `MAYA_GIT_URL` with your target GitHub repo or set `MAYA_WHEEL_URL` to a published wheel.

Next steps:
1. Change the install URL if you are using a fork or a wheel.
2. Increase or decrease `CALLS` and `CONCURRENCY` in the setup cell.
3. Download `benchmark_output.txt` from Kaggle and keep it with your notes or report.
4. If you want an even tighter public-demo profile, combine this with the repo's `--safe-mode` CLI option.